# 实验四 K-Means 聚类

K-Means 是最经典的划分聚类算法，目标是将数据集中的样本划分为若干个组（簇），使得同一簇内的样本彼此相似、不同簇的样本彼此分离。

本实验使用 `make_blobs` 生成二维数据，完成 K-Means 基础聚类，并重点修改 `n_clusters`、`init`、`n_init` 和 `cluster_std`，观察聚类图、WCSS 和轮廓系数的变化。

## 1. 生成数据并完成基础聚类

生成 3 个簇的数据，设置 `n_clusters=3`，输出聚类中心、WCSS，并绘制聚类结果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 生成 3 个簇的数据
X, y_true = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)

# 创建并训练 K-Means 模型
model = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = model.fit_predict(X)
print('聚类中心：\n', model.cluster_centers_)
print('WCSS：', model.inertia_)

# 绘制聚类结果
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis')
plt.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
            marker='X', s=200, label='聚类中心')
plt.legend()
plt.title('K-Means 聚类结果')
plt.show()

> **操作要求**：记录聚类中心和 WCSS，观察三个聚类中心是否位于对应样本簇的中间位置。

## 2. 修改聚类数 n_clusters

将 `n_clusters` 分别设置为 2、3、4，观察簇合并和簇拆分现象。

In [ ]:
for k in [2, 3, 4]:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = model.fit_predict(X)
    plt.figure()
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis')
    plt.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
                marker='X', s=180)
    plt.title(f'k = {k}, WCSS = {model.inertia_:.1f}')
    plt.show()

> **操作要求**：比较 k=2、3、4 的结果，记录聚类数过少时哪些簇被合并，聚类数过多时哪些簇被拆分。

## 3. 使用肘部法则和轮廓系数选择 k

绘制 k=1~6 的肘部曲线，并计算 k=2~6 的轮廓系数，判断较合适的 k。

In [ ]:
from sklearn.metrics import silhouette_score

k_values = range(1, 7)
wcss = []
for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X)
    wcss.append(model.inertia_)

# 肘部法则
plt.plot(k_values, wcss, marker='o')
plt.xlabel('聚类数 k')
plt.ylabel('WCSS')
plt.title('肘部法则')
plt.show()

# 轮廓系数
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    print(f'k={k}, 轮廓系数={score:.4f}')

> **操作要求**：观察 WCSS 下降开始变缓的位置，并找出轮廓系数较大的 k，结合两种结果选择聚类数。

## 4. 修改初始化方式 init 和初始化次数 n_init

比较 `init="k-means++"` 与 `init="random"`；再将 `n_init` 设置为 1、5、10，观察 WCSS 变化。

In [ ]:
# 比较两种初始化方式
for init_method in ['k-means++', 'random']:
    model = KMeans(n_clusters=3, init=init_method, n_init=1, random_state=0)
    model.fit(X)
    print(init_method, 'WCSS =', model.inertia_)

# 比较不同初始化次数
for n in [1, 5, 10]:
    model = KMeans(n_clusters=3, init='random', n_init=n, random_state=42)
    model.fit(X)
    print('n_init =', n, 'WCSS =', model.inertia_)

> **操作要求**：比较两种初始化方式的 WCSS；观察 n_init 增大后，模型选择较好聚类结果的机会是否增加。

## 5. 修改数据离散程度 cluster_std

将 `cluster_std` 设置为 0.5、1.0、2.0，观察样本重叠和聚类难度变化。

In [ ]:
for std in [0.5, 1.0, 2.0]:
    X_std, _ = make_blobs(n_samples=150, centers=3, cluster_std=std, random_state=42)
    model = KMeans(n_clusters=3, n_init=10, random_state=42)
    labels = model.fit_predict(X_std)
    plt.figure()
    plt.scatter(X_std[:, 0], X_std[:, 1], c=labels, cmap='viridis')
    plt.title(f'cluster_std = {std}')
    plt.show()

> **操作要求**：比较三组数据。记录 cluster_std 增大后，样本是否更加分散、不同簇是否更容易重叠。

## 6. 创新拓展实践：学生消费数据聚类

构造一个小规模学生消费数据集，每个样本包含"每周就餐次数"和"每周消费金额"两个特征：

任务：先设置 `n_clusters=3` 完成聚类并绘图，再将 `n_clusters` 修改为 2。比较两组结果，并根据各簇样本的就餐次数和消费金额概括学生消费特征。

> 注意：当两个特征的数值范围差异较大时，可使用 `StandardScaler` 对数据进行标准化后再聚类。

In [ ]:
student_data = np.array([
    [5, 80], [6, 95], [4, 70],
    [12, 180], [13, 200], [11, 165],
    [20, 320], [22, 350], [19, 300]
])

# 使用 StandardScaler 标准化
from sklearn.preprocessing import StandardScaler

X_stu = StandardScaler().fit_transform(student_data)

# n_clusters=3
model3 = KMeans(n_clusters=3, n_init=10, random_state=42)
labels3 = model3.fit_predict(X_stu)
print('n_clusters=3，聚类中心：\n', model3.cluster_centers_)

plt.figure()
plt.scatter(X_stu[:, 0], X_stu[:, 1], c=labels3, cmap='viridis')
plt.scatter(model3.cluster_centers_[:, 0], model3.cluster_centers_[:, 1],
            marker='X', s=200)
plt.title('学生消费数据聚类 (k=3)')
plt.show()

# n_clusters=2
model2 = KMeans(n_clusters=2, n_init=10, random_state=42)
labels2 = model2.fit_predict(X_stu)
print('n_clusters=2，聚类中心：\n', model2.cluster_centers_)

plt.figure()
plt.scatter(X_stu[:, 0], X_stu[:, 1], c=labels2, cmap='viridis')
plt.scatter(model2.cluster_centers_[:, 0], model2.cluster_centers_[:, 1],
            marker='X', s=200)
plt.title('学生消费数据聚类 (k=2)')
plt.show()